# 속성 기반 감성분석 (ABSA) — KLUE-RoBERTa

**문제 정의**: (리뷰 문장, 속성) 쌍을 입력받아 그 속성에 대한 감성(긍정/부정/중립)을 분류
**방법**: 문장쌍 분류 방식 (Sun et al., 2019) — `[CLS] 속성 [SEP] 리뷰 문장 [SEP]` → 3-class 분류
**데이터**: AIHub 속성기반 감정분석 데이터 (JSON)

실행 순서: 0 → 1 → 2(구조 확인 후 키 이름 수정) → 3 이후 끝까지

In [ ]:
# 0. 패키지 설치 (처음 한 번만, 설치 후 커널 재시작)
!pip install -q transformers scikit-learn pandas tqdm

In [ ]:
# 1. 설정
import os, glob, json, random
import numpy as np
import pandas as pd
import torch

# ▼ AIHub 데이터를 압축 해제한 폴더 경로로 바꾸세요 (하위 폴더까지 *.json 전부 탐색)
DATA_DIR   = os.path.expanduser("~/data/absa")

MODEL_NAME = "klue/roberta-base"   # 더 가볍게: "klue/bert-base" / 더 강하게: "klue/roberta-large"
MAX_LEN    = 256
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 2e-5
SEED       = 42
MAX_SAMPLES = None    # 코드 점검용으로 빠르게 돌려볼 땐 5000 같은 숫자로
SAVE_DIR   = "./absa_model"   # ⚠️ .gitignore에 추가하세요 (용량 큼)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

json_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.json"), recursive=True))
print("JSON 파일 수:", len(json_files))
print(json_files[:5])

In [ ]:
# 2. 데이터 구조 확인 — 출력을 보고 아래 3번의 키 이름을 맞추세요
with open(json_files[0], encoding="utf-8-sig") as f:
    sample = json.load(f)

def peek(obj, depth=0, max_depth=4):
    pad = "  " * depth
    if depth > max_depth: return
    if isinstance(obj, dict):
        for k, v in list(obj.items())[:15]:
            preview = repr(v)[:60] if not isinstance(v, (dict, list)) else type(v).__name__
            print(f"{pad}{k}: {preview}")
            if isinstance(v, (dict, list)): peek(v, depth + 1, max_depth)
    elif isinstance(obj, list) and obj:
        print(f"{pad}[list, len={len(obj)}] 첫 원소:")
        peek(obj[0], depth + 1, max_depth)

peek(sample)

In [ ]:
# 3. 데이터 파싱 → (text, aspect, label) 표
# ▼ 2번 출력에 맞게 키 이름 수정
TEXT_KEY      = "RawText"            # 리뷰 원문
ASPECTS_KEY   = "Aspects"            # 속성 목록
ASPECT_KEY    = "Aspect"             # 속성 이름 (예: 가격, 품질, 디자인)
POLARITY_KEY  = "SentimentPolarity"  # 감성 값 (예: -1 / 0 / 1)

POL_MAP = {-1: "부정", 0: "중립", 1: "긍정"}

def norm_polarity(v):
    try:
        return POL_MAP.get(int(float(v)), str(v))
    except (TypeError, ValueError):
        return str(v).strip()

def iter_records(obj):
    """파일이 리스트/딕셔너리/중첩 구조 어느 쪽이든 리뷰 단위 레코드를 꺼냄"""
    if isinstance(obj, list):
        for x in obj: yield from iter_records(x)
    elif isinstance(obj, dict):
        if TEXT_KEY in obj:
            yield obj
        else:
            for v in obj.values():
                if isinstance(v, (list, dict)): yield from iter_records(v)

def split_of(path):
    p = path.lower()
    if "validation" in p or "valid" in p or "검증" in p: return "valid"
    return "train"

rows, bad = [], 0
for path in json_files:
    try:
        with open(path, encoding="utf-8-sig") as f:
            obj = json.load(f)
    except Exception as e:
        bad += 1; continue
    for rec in iter_records(obj):
        text = str(rec.get(TEXT_KEY, "")).strip()
        for a in rec.get(ASPECTS_KEY) or []:
            aspect = str(a.get(ASPECT_KEY, "")).strip()
            if text and aspect and a.get(POLARITY_KEY) is not None:
                rows.append({"text": text, "aspect": aspect,
                             "label": norm_polarity(a[POLARITY_KEY]),
                             "split": split_of(path)})

df = pd.DataFrame(rows)
print(f"읽기 실패 파일: {bad} | 전체 (문장,속성) 쌍: {len(df):,}")

# 같은 문장·같은 속성에 라벨이 여러 개면 다수결, 동률(충돌)이면 제거
def resolve(g):
    vc = g["label"].value_counts()
    if len(vc) > 1 and vc.iloc[0] == vc.iloc[1]: return None
    return vc.index[0]
agg = df.groupby(["text", "aspect", "split"]).apply(resolve).dropna().reset_index(name="label")
print(f"중복·충돌 정리 후: {len(agg):,}")
df = agg

print("\n[라벨 분포]"); print(df["label"].value_counts())
print("\n[상위 속성 20개]"); print(df["aspect"].value_counts().head(20))
df.head()

In [ ]:
# 4. train / valid 분리
# AIHub의 Training/Validation 폴더 구분이 있으면 그대로 쓰고,
# 없으면 '문장 단위'로 9:1 분리 (같은 문장이 양쪽에 섞이는 누수 방지)
if (df["split"] == "valid").any() and (df["split"] == "train").any():
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    valid_df = df[df["split"] == "valid"].reset_index(drop=True)
else:
    texts = df["text"].unique()
    rng = np.random.default_rng(SEED); rng.shuffle(texts)
    valid_texts = set(texts[: int(len(texts) * 0.1)])
    is_valid = df["text"].isin(valid_texts)
    train_df, valid_df = df[~is_valid].reset_index(drop=True), df[is_valid].reset_index(drop=True)

if MAX_SAMPLES:
    train_df = train_df.sample(min(MAX_SAMPLES, len(train_df)), random_state=SEED).reset_index(drop=True)
    valid_df = valid_df.sample(min(MAX_SAMPLES // 5, len(valid_df)), random_state=SEED).reset_index(drop=True)

LABELS = sorted(df["label"].unique())
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print("labels:", label2id)
print(f"train {len(train_df):,} | valid {len(valid_df):,}")

In [ ]:
# 5. 토크나이즈 & DataLoader  —  입력: [CLS] 속성 [SEP] 문장 [SEP]
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ABSADataset(Dataset):
    def __init__(self, frame):
        self.enc = tokenizer(list(frame["aspect"]), list(frame["text"]),
                             truncation="only_second", max_length=MAX_LEN)
        self.labels = [label2id[l] for l in frame["label"]]
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

train_ds, valid_ds = ABSADataset(train_df), ABSADataset(valid_df)
collator = DataCollatorWithPadding(tokenizer)   # 배치마다 필요한 길이만큼만 패딩
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collator, num_workers=2)
valid_dl = DataLoader(valid_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator, num_workers=2)

print(tokenizer.decode(train_ds[0]["input_ids"]))

In [ ]:
# 6. 모델 · 옵티마이저 · 평가 함수
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id).to(device)

# 라벨 불균형 보정 (긍정이 압도적으로 많은 경우가 흔함)
counts = train_df["label"].map(label2id).value_counts().sort_index().values
class_weights = torch.tensor(len(train_df) / (len(LABELS) * counts), dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
print("class weights:", dict(zip(LABELS, class_weights.cpu().numpy().round(3))))

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_dl) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)

use_amp = device.type == "cuda"
amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)

@torch.no_grad()
def evaluate(dl):
    model.eval(); preds, golds = [], []
    for batch in tqdm(dl, desc="eval", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            logits = model(**batch).logits
        preds += logits.argmax(-1).tolist(); golds += labels.tolist()
    return golds, preds

In [ ]:
# 7. 학습 (에폭마다 검증 macro-F1 기준으로 best 모델 저장)
best_f1, history = -1, []
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    pbar = tqdm(train_dl, desc=f"epoch {epoch}/{EPOCHS}")
    for step, batch in enumerate(pbar, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            loss = loss_fn(model(**batch).logits.float(), labels)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item()
        if step % 50 == 0: pbar.set_postfix(loss=f"{running/step:.4f}")

    golds, preds = evaluate(valid_dl)
    acc, f1 = accuracy_score(golds, preds), f1_score(golds, preds, average="macro")
    history.append({"epoch": epoch, "train_loss": running / len(train_dl), "val_acc": acc, "val_macro_f1": f1})
    print(f"[epoch {epoch}] train_loss={running/len(train_dl):.4f}  val_acc={acc:.4f}  val_macroF1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained(SAVE_DIR); tokenizer.save_pretrained(SAVE_DIR)
        print(f"  ↳ best 모델 저장 ({SAVE_DIR})")

pd.DataFrame(history)

In [ ]:
# 8. 최종 평가 (best 모델 기준)
model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device)
golds, preds = evaluate(valid_dl)
print(classification_report(golds, preds, target_names=LABELS, digits=4))

# 속성별 정확도 (데이터가 많은 상위 15개 속성)
valid_df["pred"] = [id2label[p] for p in preds]
by_aspect = (valid_df.assign(correct=valid_df["pred"] == valid_df["label"])
             .groupby("aspect")["correct"].agg(["mean", "count"])
             .sort_values("count", ascending=False).head(15))
by_aspect.rename(columns={"mean": "accuracy"})

In [ ]:
# 9. 오답 사례 확인 (보고서 '오류 분석'에 활용)
wrong = valid_df[valid_df["pred"] != valid_df["label"]]
pd.set_option("display.max_colwidth", 120)
wrong.sample(min(15, len(wrong)), random_state=SEED)[["aspect", "label", "pred", "text"]]

In [ ]:
# 10. 새 문장 추론
@torch.no_grad()
def predict(text, aspects):
    model.eval()
    enc = tokenizer(aspects, [text] * len(aspects), truncation="only_second",
                    max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
    probs = torch.softmax(model(**enc).logits.float(), -1).cpu().numpy()
    return pd.DataFrame([{"aspect": a, "pred": id2label[int(p.argmax())],
                          **{l: round(float(p[i]), 3) for i, l in id2label.items()}}
                         for a, p in zip(aspects, probs)])

predict("배송은 엄청 빨랐는데 가격 대비 품질이 너무 별로예요. 디자인은 예쁘네요.",
        ["배송", "가격", "품질", "디자인"])